# Step 4: Model Training and Forecasting

In this step, we train forecasting models using the feature-engineered dataset.
A time-aware train–test split is used to avoid data leakage.

## Step 4.0: Initialization

This step imports all required libraries and prepares the environment
for model training and evaluation.


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Initialization completed successfully")


Initialization completed successfully


### Interpretation

All required libraries for data handling, modeling, and evaluation
have been successfully imported.


## Step 4.1: Feature and Target Selection

In this step, the dataset is split into input features (X)
and the target variable (y).


## Step A: Create Clean Feature Dataset

This step removes rows with missing values created by lag and rolling features.
The resulting dataset will be used for model training.


In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("../DATASET/Uber-Jan-Feb-FOIL.csv")

# Convert date column to datetime and set index
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df = df.sort_index()

# Create feature engineering base
df_fe = df[['trips', 'active_vehicles']].copy()

# Time-based features
df_fe['day'] = df_fe.index.day
df_fe['day_of_week'] = df_fe.index.dayofweek
df_fe['week_of_year'] = df_fe.index.isocalendar().week.astype(int)
df_fe['month'] = df_fe.index.month
df_fe['is_weekend'] = df_fe['day_of_week'].isin([5, 6]).astype(int)

# Lag features
df_fe['trips_lag_1'] = df_fe['trips'].shift(1)
df_fe['trips_lag_7'] = df_fe['trips'].shift(7)
df_fe['trips_lag_14'] = df_fe['trips'].shift(14)

# Rolling features
df_fe['trips_roll_7'] = df_fe['trips'].rolling(window=7).mean()
df_fe['trips_roll_14'] = df_fe['trips'].rolling(window=14).mean()
df_fe['trips_std_7'] = df_fe['trips'].rolling(window=7).std()

print("Recovery step executed successfully")
print("df shape:", df.shape)
print("df_fe shape:", df_fe.shape)


Recovery step executed successfully
df shape: (354, 3)
df_fe shape: (354, 13)


### Interpretation

- The original dataset (`df`) has been reloaded and indexed by date.
- All feature engineering steps have been rebuilt in a single controlled step.
- The dataframe `df_fe` now contains raw, time-based, lag, and rolling features.
- The environment is now stable for further model training steps.


## Step 1: Handle Missing Values

Lag and rolling features introduce missing values.
These rows must be removed before model training.


In [3]:
df_fe_clean = df_fe.dropna()

print("df_fe_clean created successfully")
print("Clean dataset shape:", df_fe_clean.shape)


df_fe_clean created successfully
Clean dataset shape: (340, 13)


### Interpretation

- Rows with missing values have been removed.
- The cleaned dataset is safe for model training.


## Step 2: Prepare Features and Target

This step separates input features and the target variable.


In [4]:
X = df_fe_clean.drop(columns=['trips'])
y = df_fe_clean['trips']

print("Features and target prepared successfully")
print("X shape:", X.shape)
print("y shape:", y.shape)


Features and target prepared successfully
X shape: (340, 12)
y shape: (340,)


### Interpretation

- X contains all engineered features.
- y contains the daily Uber trip demand.
- The data is ready for train–test splitting.



## Step 4.2: Train–Test Split (Time-Aware)

In this step, the dataset is split into training and testing sets
while preserving the chronological order of the data.

This approach prevents data leakage, which is critical for
time-series forecasting problems.


In [5]:
# Time-aware train-test split (80% train, 20% test)
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Step 4.2 executed successfully")
print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)


Step 4.2 executed successfully
Training feature shape: (272, 12)
Testing feature shape: (68, 12)
Training target shape: (272,)
Testing target shape: (68,)


### Interpretation

- The first 80% of the data is used for training the model.
- The remaining 20% of the data is reserved for testing.
- Data is split chronologically, ensuring that future information
  is not used to predict the past.
- This split strategy makes the evaluation realistic and reliable
  for forecasting tasks.


## Step 4.3: Baseline Model Training (Linear Regression)

Linear Regression is used as a baseline model to establish
a reference level of performance for trip demand forecasting.


In [6]:
from sklearn.linear_model import LinearRegression

# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Step 4.3 executed successfully: Linear Regression model trained")


Step 4.3 executed successfully: Linear Regression model trained


## Step 4.4: Evaluation of Linear Regression Model

In this step, the performance of the Linear Regression model
is evaluated using standard regression metrics:
MAE, RMSE, and R².


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions on test data
y_pred_lr = lr_model.predict(X_test)

# Calculate evaluation metrics
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Step 4.4 executed successfully: Linear Regression evaluated")
print("MAE:", round(mae_lr, 2))
print("RMSE:", round(rmse_lr, 2))
print("R²:", round(r2_lr, 3))


Step 4.4 executed successfully: Linear Regression evaluated
MAE: 1208.51
RMSE: 1712.9
R²: 0.976


### Interpretation

- MAE represents the average absolute error in predicted trip counts.
- RMSE penalizes larger prediction errors more strongly than MAE.
- R² indicates how well the model explains variability in trip demand.
- These values serve as a baseline benchmark for comparison
  with more advanced models.


## Step 4.5: Advanced Model Training (Random Forest Regressor)

Random Forest is an ensemble-based model that can capture
non-linear relationships and complex interactions between features.
This model is expected to outperform the linear baseline.


In [8]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Step 4.5 executed successfully: Random Forest model trained")


Step 4.5 executed successfully: Random Forest model trained


### Interpretation

- The Random Forest model has been trained using multiple decision trees.
- It can capture non-linear patterns in trip demand that
  Linear Regression cannot model.
- This model is expected to provide improved forecasting accuracy,
  especially with lag and rolling features.


## Step 4.6: Evaluation of Random Forest Model

In this step, the performance of the Random Forest model
is evaluated using the same metrics as the baseline model
to enable a fair comparison.


In [9]:
# Make predictions on test data
y_pred_rf = rf_model.predict(X_test)

# Calculate evaluation metrics
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Step 4.6 executed successfully: Random Forest evaluated")
print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))
print("R²:", round(r2_rf, 3))


Step 4.6 executed successfully: Random Forest evaluated
MAE: 935.89
RMSE: 1512.87
R²: 0.982


## Step 4.7: Model Comparison

This step compares the performance of the baseline Linear Regression model
and the advanced Random Forest model using common evaluation metrics.


In [10]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R²": [r2_lr, r2_rf]
})

print("Step 4.7 executed successfully: Model comparison table created")
comparison_df


Step 4.7 executed successfully: Model comparison table created


,Model,MAE,RMSE,R²
0,Linear Regression,1208.511391,1712.899143,0.976411
1,Random Forest,935.894902,1512.874356,0.981599


### Interpretation

- The Random Forest model outperforms the Linear Regression model
  across all evaluation metrics.
- Lower MAE and RMSE values indicate more accurate predictions.
- Higher R² shows better explanation of variability in trip demand.
- This confirms that non-linear models benefit from the engineered
  lag and rolling features.
